# ch04 LSTM 与 GRU（练习版）

> 补全所有 `____`，让每个代码块末尾的 `assert` 全绿。
> 数据：`data/load_curve.csv` 的 A 台区（居民型），2025 全年 8760 个整点 —— 与 ch03 同口径。
>
> **本章最容易卡住的三个地方**，卡住时重点回看提示：
> 1. LSTM 的 gate 顺序（`i, f, g, o`）
> 2. GRU 的 `n` 门偏置（`b_hh` 在括号里）
> 3. `h_n` / `c_n` 的形状
>
> 手写对账的 assert 都是 **`< 1e-12`**（double 精度下的逐值一致）。
> **对不上就是有 bug**，别放宽阈值 —— 放宽之后你就学不到任何东西了。

## 任务清单

| 任务 | 主题 | 挖空 | 对应讲解小节 |
|---|---|---|---|
| 1 | 数据准备（与 ch03 同口径） | 11 | 4.1 |
| 2 | 记忆保留率：`tanh` 通路 vs 加法通路 | 2 | 4.1 |
| 3 | 手写 LSTM 单步，与 `nn.LSTMCell` 对账 | 11 | 4.2 / 4.3 |
| 4 | 手写 LSTM 全序列，与 `nn.LSTM` 对账 | 15 | 4.3 |
| 5 | gate 顺序写错的两种下场 | 16 | 4.3.1 |
| 6 | 手写 GRU 全序列，与 `nn.GRU` 对账 | 12 | 4.4 |
| 7 | GRU 的 `n` 门偏置坑 | 5 | 4.4.1 |
| 8 | `bidirectional` / `num_layers` 的形状 | 6 | 4.5 |
| 9 | 四个模型的 MAE 对照表 | 7 | 4.5 |
| 10 | 梯度流对照：RNN vs LSTM | 4 | 4.6 |
| 11 | 遗忘门偏置扫描 | 2 | 4.6.1 |
| **合计** | 11 个任务 / 26 个 `@@todo` 块 | **91** | |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

DATA = Path("data")          # notebook 的 cwd = coding/05_timeseries
CSV = DATA / "load_curve.csv"

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

raw = pd.read_csv(CSV, encoding="utf-8-sig", parse_dates=["时间戳"])
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "（本章全程 CPU）")
print("原始行数", len(raw), "| 列", list(raw.columns))

In [ ]:
df = raw.drop_duplicates().sort_values("时间戳").reset_index(drop=True)
print("去重排序后", len(df), "行")

# 与 ch03 完全同一份主序列：A 台区（居民型），2025 全年 8760 个整点
a_frame = df[df["台区编号"] == "STATION_A_01"].set_index("时间戳").sort_index()
s_a = a_frame["负荷值"].asfreq("h").loc["2025-01-01":"2025-12-31 23:00"]
print("A 台区整点 %d | 缺失 %d" % (len(s_a), int(s_a.isna().sum())))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# macOS 上 matplotlib 默认字体不含中文，不设置的话图上全是方块
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "STHeiti"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib", matplotlib.__version__, "| 中文字体 PingFang SC")

In [ ]:
# 脚手架：下面的常量与工具函数两版都有，你只填 @@todo 块里的内容
W = 24            # 滑窗长度（与 ch03 一致）
HID = 12          # 隐藏单元数
EPOCHS = 15
BATCH = 128
SEED = 42


def clean_series(s):
    """按**小时分组**的中位数做 3 倍阈值判异常，再线性插值。"""
    med = s.groupby(s.index.hour).transform("median")
    return s.where(s.between(0, 3 * med)).interpolate()


def make_windows(z, w):
    """一维序列 → 监督矩阵：X[i] = z[i : i+w]，y[i] = z[i+w]。"""
    x = np.lib.stride_tricks.sliding_window_view(z, w)[:-1]
    return x, z[w:]


def make_rnn(rho, hidden=16, seed=1):
    """造一个 W_h 谱半径被钉死在 rho 上的 torch RNN（与 ch03 的 3.5 同口径）。"""
    with torch.random.fork_rng():
        torch.manual_seed(seed)
        m = nn.RNN(1, hidden, batch_first=True).double()
    with torch.no_grad():
        w = m.weight_hh_l0
        m.weight_hh_l0.copy_(w * (rho / float(torch.linalg.eigvals(w).abs().max())))
    return m


def make_lstm(f_bias, hidden=16, seed=1):
    """造一个 LSTM，并把**遗忘门**的偏置设成 f_bias（None = 保持默认）。

    bias_ih_l0 的第 2 段是遗忘门 —— 这就是 4.3.1 讲的 gate 顺序 i, f, g, o。
    PyTorch 默认把四门的偏置都初始化在 U(-1/sqrt(H), 1/sqrt(H))，
    所以 f = sigmoid(~0) ~= 0.5：**每过一步，记忆就被砍掉一半**。
    """
    with torch.random.fork_rng():
        torch.manual_seed(seed)
        m = nn.LSTM(1, hidden, batch_first=True).double()
    if f_bias is not None:
        with torch.no_grad():
            m.bias_ih_l0[hidden:2 * hidden] = f_bias
            m.bias_hh_l0[hidden:2 * hidden] = 0.0
    return m


def last_step_grads(model, length, seed=3):
    """loss 只挂在**最后一步**的输出上，回传得到每个输入步的梯度幅度。

    RNN / LSTM / GRU 都能用：`out, _ = model(x)` 对三者都成立
    （LSTM 的第二个返回值是 (h, c) 元组，用 `_` 接住即可）。
    """
    with torch.random.fork_rng():
        torch.manual_seed(seed)
        x = torch.randn(1, length, 1, dtype=torch.float64, requires_grad=True)
    out, _ = model(x)
    g = torch.autograd.grad(out[:, -1, :].pow(2).sum(), x)[0]
    return g[0, :, 0].abs().detach().numpy()

## 任务 1：数据准备（4.1）

In [ ]:
# TODO(1)：清洗 A 台区序列（脚手架已有 clean_series）
s_a_clean = ____

SERIES = s_a_clean.values.astype("float64")
print("主序列 %d 点 | 均值 %.6f | 标准差 %.6f" % (len(SERIES), SERIES.mean(), SERIES.std()))

# TODO(2)：按时间 8:2 切分，标准化参数只取自训练段
# 提示：和 ch03 完全同一套口径，保证两章的 MAE 可比
cut_pt = ____
MU = ____
SD = ____

Z = (SERIES - MU) / SD

# TODO(3)：切窗 + 按样本切分 + 转成 torch 张量（特征维补成 1）
# 提示：样本级切分点 = cut_pt - W；张量形状 (N, T, 1)
Xz, yz = ____
n_train = ____
Xtr, ytr = ____
Xte, yte = ____
Xtr_t = ____
ytr_t = ____
Xte_t = ____

mae_persist = float(np.abs(Xte[:, -1] - yte).mean())
print("训练 %s / 测试 %s | MU %.6f SD %.6f" % (Xtr_t.shape, Xte_t.shape, MU, SD))
print("朴素持续基线（拿窗口末格当预测）MAE %.6f" % mae_persist)

# ---- 验收 ----
assert len(SERIES) == 8760 and n_train == 6984
assert abs(MU - 454.561801) < 1e-6 and abs(SD - 72.623128) < 1e-6
assert tuple(Xtr_t.shape) == (6984, 24, 1) and tuple(Xte_t.shape) == (1752, 24, 1)
assert abs(mae_persist - 0.414592) < 1e-6, "与 ch03 同一份数据、同一个基线"

## 任务 2：记忆保留率 —— `tanh` 通路 vs 加法通路（4.1）

In [ ]:
STEPS = 40

# RNN 式记忆：唯一通道 h 每步都要穿过 tanh
pulse_rnn = [1.0]
for _ in range(STEPS):
    # TODO(4)：每步 c = tanh(0.9 * c)：0.9 是"衰减"，tanh 是"压缩"
    ____

# LSTM 式记忆：cell state 走加法通路，f 可以学到 0.99、i * g 可以学到 0
pulse_lstm = [1.0]
for _ in range(STEPS):
    # TODO(5)：加法通路：c = f * c + i * g，取 f = 0.99、i * g = 0
    ____

print("第 1 步 t=0 注入一个脉冲 1.0，看它还能剩多少：")
print("  RNN 式（每步过 tanh）   t=1 %.6f  t=5 %.6f  t=40 %.10f"
      % (pulse_rnn[1], pulse_rnn[5], pulse_rnn[40]))
print("  LSTM 式（加法通路）     t=1 %.6f  t=5 %.6f  t=40 %.10f"
      % (pulse_lstm[1], pulse_lstm[5], pulse_lstm[40]))
print("  40 步后的保留率：RNN %.6f%% vs LSTM %.6f%%（差 %.0f 倍）"
      % (100 * pulse_rnn[40], 100 * pulse_lstm[40], pulse_lstm[40] / pulse_rnn[40]))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.arange(STEPS + 1), pulse_rnn, marker="o", ms=3, label="RNN 式：c=tanh(0.9c)")
ax.plot(np.arange(STEPS + 1), pulse_lstm, marker="s", ms=3, label="LSTM 式：c=0.99c+i*g")
ax.set_xlabel("时间步")
ax.set_ylabel("记忆强度（初始脉冲 1.0）")
ax.set_title("为什么需要加法通路：40 步之后还剩多少")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

# ---- 验收 ----
assert abs(pulse_rnn[40] - 0.0074665174) < 1e-9, "RNN 式记忆 40 步后只剩 0.75%"
assert abs(pulse_lstm[40] - 0.6689717586) < 1e-9, "0.99 的 40 次方 = 66.90%"
assert pulse_lstm[40] > 50 * pulse_rnn[40]

## 任务 3：手写 LSTM 单步并对账（4.2 / 4.3）

In [ ]:
# 单步对账：B=2 个样本、D=3 维输入、H=4 个隐藏单元
B_L, D_L, H_L = 2, 3, 4

with torch.random.fork_rng():
    torch.manual_seed(5)
    x_step = torch.randn(B_L, D_L, dtype=torch.float64)
    h0 = torch.randn(B_L, H_L, dtype=torch.float64) * 0.5
    c0 = torch.randn(B_L, H_L, dtype=torch.float64) * 0.5
    cell = nn.LSTMCell(D_L, H_L).double()

h_ref, c_ref = cell(x_step, (h0, c0))

# TODO(6)：把 LSTMCell 的权重搬进 numpy（weight_ih / weight_hh，两个 bias 相加）
Wi_L = ____
Wh_L = ____
b_L = ____

# TODO(7)：一次算完四个门的预激活，再按 PyTorch 的顺序切成 i / f / g / o
# 提示：顺序是 i, f, g, o —— 不是 f, i, o, g（见 4.3.1）
gates = ____
g_i, g_f, g_g, g_o = ____

# TODO(8)：激活：i / f / o 走 sigmoid，候选记忆 g 走 tanh
i_t = ____
f_t = ____
o_t = ____
g_t = ____

# TODO(9)：两个状态方程：cell state 走加法，h 是 cell state 过 tanh 后再被 o 门筛一次
c_np = ____
h_np = ____

print("门的均值：i %.4f | f %.4f | g %+.4f | o %.4f" % (i_t.mean(), f_t.mean(), g_t.mean(), o_t.mean()))
print("LSTMCell 单步：h max|diff| %.3e | c max|diff| %.3e"
      % (np.abs(h_np - h_ref.detach().numpy()).max(), np.abs(c_np - c_ref.detach().numpy()).max()))

# ---- 验收 ----
assert np.abs(h_np - h_ref.detach().numpy()).max() < 1e-12
assert np.abs(c_np - c_ref.detach().numpy()).max() < 1e-12
assert abs(float(f_t.mean()) - 0.4730) < 1e-3, "默认初始化下遗忘门约 0.47 —— 每步丢掉一半记忆"

## 任务 4：手写 LSTM 全序列并对账（4.3）

In [ ]:
B, T_, D, H = 2, 6, 3, 5

with torch.random.fork_rng():
    torch.manual_seed(11)
    x_lstm = torch.randn(B, T_, D, dtype=torch.float64)
    lstm = nn.LSTM(D, H, batch_first=True).double()

out_l, (hn_l, cn_l) = lstm(x_lstm)
print("nn.LSTM 的 out %s | h_n %s | c_n %s"
      % (tuple(out_l.shape), tuple(hn_l.shape), tuple(cn_l.shape)))

# TODO(10)：把权重搬进 numpy
Wi = ____
Wh = ____
b = ____
xn = ____

# TODO(11)：初始化两个状态：LSTM 有 h 和 c **两条**独立通道（RNN 只有 h 一条）
h_np_l = ____
c_np_l = ____
hs_l, cs_l = ____

for t in range(T_):
    # TODO(12)：四门 + 两个状态方程，逐时间步展开
    g = ____
    gi, gf, gg, go = ____
    i_t = ____
    f_t = ____
    o_t = ____
    g_t = ____
    c_np_l = ____
    h_np_l = ____
    hs_l.append(h_np_l)
    cs_l.append(c_np_l)

hs_l = np.stack(hs_l).transpose(1, 0, 2)
cs_l = np.stack(cs_l).transpose(1, 0, 2)

d_h = float(np.abs(hs_l - out_l.detach().numpy()).max())
d_c = float(np.abs(cs_l[:, -1] - cn_l.detach().numpy()[0]).max())
print("手写 LSTM 前向 vs nn.LSTM：全部时间步 max|diff| = %.3e" % d_h)
print("手写 c_T vs nn.LSTM 的 c_n：max|diff| = %.3e" % d_c)

# ---- 验收 ----
assert hs_l.shape == (B, T_, H) and cs_l.shape == (B, T_, H)
assert d_h < 1e-12, "双精度下手写 LSTM 前向必须与 nn.LSTM 逐值一致"
assert d_c < 1e-12
assert np.abs(hs_l[:, -1] - hn_l.detach().numpy()[0]).max() < 1e-12

## 任务 5：gate 顺序写错的两种下场（4.3.1）

In [ ]:
# ---- 错误示范一：按 (f, i, o, g) 切门（听起来更符合公式书写顺序） ----
h_bad_l = np.zeros((B, H))
c_bad_l = np.zeros((B, H))
hs_bad = []
for t in range(T_):
    # TODO(13)：同样四条公式，但把 gate 顺序写成 (f, i, o, g)
    g = ____
    gf, gi, go, gg = ____
    i_t = ____
    f_t = ____
    o_t = ____
    g_t = ____
    c_bad_l = ____
    h_bad_l = ____
    hs_bad.append(h_bad_l)
hs_bad = np.stack(hs_bad).transpose(1, 0, 2)

# ---- 错误示范二：只把 i 与 f 换位，其余全对（最隐蔽的一种写法） ----
h_sw = np.zeros((B, H))
c_sw = np.zeros((B, H))
hs_sw = []
for t in range(T_):
    # TODO(14)：切门顺序是对的，但把 sigmoid 结果对调了：i 用了 f 的预激活，反之亦然
    g = ____
    gi, gf, gg, go = ____
    i_t = ____
    f_t = ____
    o_t = ____
    g_t = ____
    c_sw = ____
    h_sw = ____
    hs_sw.append(h_sw)
hs_sw = np.stack(hs_sw).transpose(1, 0, 2)

d_bad = float(np.abs(hs_bad - out_l.detach().numpy()).max())
d_sw = float(np.abs(hs_sw - out_l.detach().numpy()).max())
print("错误顺序 (f,i,o,g)：h max|diff| = %.3e" % d_bad)
print("只把 i / f 对调  ：h max|diff| = %.3e" % d_sw)

# ---- 验收 ----
assert d_bad > 0.3, "顺序错了，结果差得离谱，而且不会报任何错"
assert d_sw > 0.1, "只对调两个门的 sigmoid，一样是错的"
assert np.abs(hs_bad[:, -1] - hs_sw[:, -1]).max() > 1e-3, "两种错法错得还不一样"

## 任务 6：手写 GRU 全序列并对账（4.4）

In [ ]:
with torch.random.fork_rng():
    torch.manual_seed(12)
    x_gru = torch.randn(B, T_, D, dtype=torch.float64)
    gru = nn.GRU(D, H, batch_first=True).double()

out_g, hn_g = gru(x_gru)
print("nn.GRU 的 out %s | h_n %s（GRU 只有 h，没有 c）"
      % (tuple(out_g.shape), tuple(hn_g.shape)))

# TODO(15)：按 PyTorch 的 (r, z, n) 三段切开输入权重、循环权重，以及**两个**偏置
# 提示：np.split(张量, 3, axis=0)，偏置按 3 段切
Wir, Wiz, Win = ____
Whr, Whz, Whn = ____
bir, biz, bin_ = ____
bhr, bhz, bhn = ____
xng = ____

# TODO(16)：初始化隐状态（GRU 只有一条通道）
h_np_g = ____
hs_g = ____

for t in range(T_):
    # TODO(17)：三段公式：重置门 r（两个偏置都加）、更新门 z（同）、候选 n（b_hh 被 r 乘着）
    # 提示：n 门只直连 b_ih 那一段；b_hh 那一段在括号里，被 r_t 乘一遍（见 4.4.1）
    xt = ____
    r_t = ____
    z_t = ____
    n_t = ____
    h_np_g = ____
    hs_g.append(h_np_g)

hs_g = np.stack(hs_g).transpose(1, 0, 2)
d_gru = float(np.abs(hs_g - out_g.detach().numpy()).max())
print("手写 GRU 前向 vs nn.GRU：max|diff| = %.3e" % d_gru)
print("手写 h_T vs nn.GRU 的 h_n：max|diff| = %.3e"
      % np.abs(hs_g[:, -1] - hn_g.detach().numpy()[0]).max())

# ---- 验收 ----
assert d_gru < 1e-12, "双精度下手写 GRU 前向必须与 nn.GRU 逐值一致"
assert hs_g.shape == (B, T_, H)

## 任务 7：GRU 的 `n` 门偏置坑（4.4.1）

In [ ]:
# ---- 错误示范：以为"三个门都能直接加 b_ih + b_hh" ----
allb = (gru.bias_ih_l0 + gru.bias_hh_l0).detach().numpy()
h_bad_g = np.zeros((B, H))
hs_bad_g = []
for t in range(T_):
    # TODO(18)：用"三个门对称"的写法：r / z / n 都直接加 bi + bh 的对应段
    # 提示：这是错的 —— 复现它，别改对
    xt = ____
    r_t = ____
    z_t = ____
    n_t = ____
    h_bad_g = ____
    hs_bad_g.append(h_bad_g)
hs_bad_g = np.stack(hs_bad_g).transpose(1, 0, 2)

d_bad_g = float(np.abs(hs_bad_g - out_g.detach().numpy()).max())
print("n 门的 b_hh 处理错：max|diff| = %.4f" % d_bad_g)
print("偏差达到输出量级的 %.1f%% —— 又是一个不报错、只是悄悄变差的写法"
      % (100 * d_bad_g / float(np.abs(out_g.detach().numpy()).max())))

# ---- 验收 ----
assert d_bad_g > 0.05, "n 门的 b_hh 必须被 r 门乘着走"
assert abs(d_bad_g - 0.1552) < 5e-3

## 任务 8：`bidirectional` / `num_layers` 的形状（4.5）

In [ ]:
SHAPE_CASES = [(1, False), (2, False), (1, True), (2, True)]
shape_l = {}
shape_g = {}
x_shape = torch.zeros(4, 10, 1, dtype=torch.float64)

for nl, bd in SHAPE_CASES:
    # TODO(19)：造 LSTM(num_layers=nl, bidirectional=bd)，记录 out / h / c 的形状
    # 提示：out 永远是 (B, T, H * 方向数)；h / c 是 (层数 * 方向数, B, H)
    m_l = ____
    o_l, (h_l, c_l) = ____
    shape_l[(nl, bd)] = ____

    # TODO(20)：同样的 GRU（注意 GRU 没有 c）
    m_g = ____
    o_g, h_g = ____
    shape_g[(nl, bd)] = ____

print("输入 (4, 10, 1)，隐藏 8：")
for nl, bd in SHAPE_CASES:
    print("  layers=%d bidir=%-5s | LSTM out %s h %s c %s | GRU out %s h %s"
          % (nl, bd, shape_l[(nl, bd)][0], shape_l[(nl, bd)][1], shape_l[(nl, bd)][2],
             shape_g[(nl, bd)][0], shape_g[(nl, bd)][1]))

# ---- 验收 ----
assert shape_l[(1, False)] == ((4, 10, 8), (1, 4, 8), (1, 4, 8))
assert shape_l[(2, True)] == ((4, 10, 16), (4, 4, 8), (4, 4, 8))
assert shape_g[(2, True)] == ((4, 10, 16), (4, 4, 8))
assert shape_l[(1, True)][1][0] == 2, "双向: 第 0 维 = 层数 x 2"
assert shape_l[(2, False)][0][2] == 8, "堆叠层数不改隐藏维，只改 h 的第 0 维"

## 任务 9：四个模型的 MAE 对照表（4.5）

In [ ]:
class SeqRegressor(nn.Module):
    """只换循环层（RNN / LSTM / GRU），其余完全一致。"""

    def __init__(self, kind="lstm", hidden=HID, layers=1, bidir=False):
        super().__init__()
        cls = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[kind]
        self.r = cls(1, hidden, num_layers=layers, batch_first=True, bidirectional=bidir)
        self.fc = nn.Linear(hidden * (2 if bidir else 1), 1)

    def forward(self, x):
        out, _ = self.r(x)
        return self.fc(out[:, -1, :])


def train_regressor(kind, **kw):
    """统一口径：同一个 seed、同一组超参，只换循环层。"""
    with torch.random.fork_rng():
        torch.manual_seed(SEED)
        model = SeqRegressor(kind, **kw)
        opt = torch.optim.Adam(model.parameters(), lr=1e-2)
        lossf = nn.MSELoss()
        for epoch in range(EPOCHS):
            perm = torch.randperm(len(Xtr_t))
            for i in range(0, len(Xtr_t), BATCH):
                idx = perm[i: i + BATCH]
                # TODO(21)：一个 batch 的四行：清零 → 前向算 loss → 反向 → 更新
                ____
                loss = ____
                ____
                ____
        model.eval()
        with torch.no_grad():
            # TODO(22)：测试段前向，并把 (N, T, 1) 压平成 (N,)
            pred = ____
    # TODO(23)：返回测试段 MAE（标准化空间）
    mae_z = ____
    return mae_z


CFG = [("rnn", {}), ("lstm", {}), ("gru", {}), ("lstm", {"bidir": True})]
mae_table = {}
for kind, kw in CFG:
    # TODO(24)：按统一口径训练一个模型，把 MAE 记进 mae_table
    mae_table[(kind, tuple(sorted(kw.items())))] = ____

print("朴素持续基线 MAE %.6f（%.2f kW）" % (mae_persist, mae_persist * SD))
for kind, kw in CFG:
    m = mae_table[(kind, tuple(sorted(kw.items())))]
    print("%-5s %-24s 测试 MAE %.6f（%.2f kW）  相对朴素 %+.1f%%"
          % (kind, kw, m, m * SD, 100 * (m / mae_persist - 1)))

# ---- 验收 ----
assert abs(mae_table[("rnn", ())] - 0.259707) < 1e-4
assert abs(mae_table[("lstm", ())] - 0.215303) < 1e-4
assert abs(mae_table[("gru", ())] - 0.223200) < 1e-4
assert abs(mae_table[("lstm", (("bidir", True),))] - 0.225713) < 1e-4
assert all(v < mae_persist for v in mae_table.values()), "四个模型都要打赢朴素基线"

## 任务 10：梯度流对照 —— RNN vs LSTM（4.6）

In [ ]:
T_LONG = 160          # 与 ch03 的 3.5 同口径：序列长度 160

# TODO(25)：同一口径量两个模型的「第 1 步 / 第 T 步」梯度比
# 提示：RNN 用 make_rnn(0.9)（谱半径 0.9）；LSTM 用 make_lstm(None)（PyTorch 默认初始化）
g_rnn = ____
g_lstm = ____
ratio_rnn = ____
ratio_lstm = ____

print("T=%d：RNN(rho=0.9) 首末比 %.3e" % (T_LONG, ratio_rnn))
print("T=%d：LSTM(默认初始化) 首末比 %.3e" % (T_LONG, ratio_lstm))
print("差 %.1f 个数量级 —— 但方向是**反的**：门控默认配置下更差！"
      % (np.log10(ratio_rnn) - np.log10(ratio_lstm)))

# ---- 验收 ----
assert abs(np.log10(ratio_rnn) - (-12.017)) < 0.5, "与 ch03 的 9.615e-13 对得上"
assert ratio_lstm < ratio_rnn, "负结果：LSTM 默认初始化下梯度衰减比 RNN 更快"
assert np.log10(ratio_lstm) < -30

## 任务 11：遗忘门偏置扫描（4.6.1）

In [ ]:
F_BIASES = [None, 1.0, 2.0, 3.0]

ratio_f = {}
for fb in F_BIASES:
    # TODO(26)：造一个遗忘门偏置为 fb 的 LSTM，量 T=160 上的首末梯度比
    # 提示：脚手架 make_lstm(f_bias)，None 表示保持 PyTorch 默认初始化
    g_f = ____
    ratio_f[fb] = ____

for fb in F_BIASES:
    r = ratio_f[fb]
    print("遗忘门偏置 %-5s → 首末比 %.3e   平均每步 x%.6f"
          % (str(fb), r, r ** (1 / (T_LONG - 1))))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
labels = ["默认\n(f=0.5)", "f 偏置=1\n(f=0.73)", "f 偏置=2\n(f=0.88)", "f 偏置=3\n(f=0.95)"]
axes[0].bar(np.arange(4), [np.log10(ratio_f[fb]) for fb in F_BIASES], color="tab:green")
axes[0].axhline(np.log10(ratio_rnn), color="tab:blue", ls="--", lw=1.6,
                label="RNN(rho=0.9) 参照线")
axes[0].axhline(np.log10(ratio_lstm), color="tab:red", ls=":", lw=1.6,
                label="LSTM 默认 参照线")
axes[0].set_xticks(np.arange(4))
axes[0].set_xticklabels(labels)
axes[0].set_ylabel("log10(第 1 步 / 第 T 步 梯度比)")
axes[0].set_title("遗忘门越接近 1，梯度衰减越慢（T=160）")
axes[0].legend(fontsize=8)
axes[1].plot(np.arange(T_LONG), np.log10(last_step_grads(make_lstm(3.0), T_LONG)), lw=1.6,
             label="LSTM f 偏置=3")
axes[1].plot(np.arange(T_LONG), np.log10(last_step_grads(make_rnn(0.9), T_LONG)), lw=1.6,
             label="RNN rho=0.9")
axes[1].set_xlabel("输入步序号（0 = 最早）")
axes[1].set_ylabel("log10(梯度幅度)")
axes[1].set_title("逐步梯度曲线：LSTM 的尾部是一条斜线，RNN 是断崖")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()

# ---- 验收 ----
assert ratio_f[3.0] > ratio_f[2.0] > ratio_f[1.0] > ratio_f[None], \
    "遗忘门偏置越大（f 越接近 1），越能保住远期梯度"
assert abs(np.log10(ratio_f[3.0]) - (-0.5223)) < 0.5
assert ratio_f[3.0] > 1e10 * ratio_lstm, "把 f 顶上去之后，比默认配置好了 10 个数量级以上"

## 自查清单（能不看笔记答出来才算过）

- [ ] 11 个代码块的 assert 全部通过
- [ ] RNN 的 `h` 一条通道为什么不够用？40 步后 RNN 式还剩多少、加法通路还剩多少？
- [ ] LSTM 的四个门分别叫什么？各自用什么激活函数？值域是多少？
- [ ] `c_t = f ⊙ c_{t-1} + i ⊙ g` 这一行为什么被称为"加法高速公路"？
- [ ] PyTorch 的 LSTM gate 顺序是什么？写错之后 `h` 会差多少？
- [ ] 只把 `i` 与 `f` 对调会怎样？为什么不报错？
- [ ] GRU 的段序是什么？`n` 门的 `b_hh` 为什么要被 `r` 乘？
- [ ] LSTM 和 GRU 的偏置处理有什么不同？
- [ ] `out` / `h_n` / `c_n` 三个形状分别是什么？双向和堆叠各改哪一维？
- [ ] 为什么双向 LSTM 不能用于在线预测？
- [ ] 本章的负结果是什么？为什么 LSTM 默认初始化下梯度比 RNN 衰减更快？
- [ ] 遗忘门偏置设成 3 之后，T=160 的首末梯度比是多少？
- [ ] 为什么"把遗忘门偏置顶上去"这件事零成本、几乎无副作用？

## 关键真值对照（做完再对）

| 量 | 值 |
|---|---|
| 序列长度 / 训练样本 / 测试样本 | 8760 / 6984 / 1752 |
| MU / SD | 454.561801 / 72.623128 |
| 朴素持续基线 MAE | 0.414592 |
| RNN 式记忆 40 步后（初始脉冲 1.0） | 0.0074665174（保留 0.747%） |
| 加法通路 40 步后 | 0.6689717586（保留 66.90%） |
| 默认遗忘门均值 `f` | 0.4730 |
| 手写单步 vs `nn.LSTMCell` max&#124;diff&#124;（h / c） | 5.55e-17 / 5.55e-17 |
| 手写全序列 vs `nn.LSTM` max&#124;diff&#124;（out / c_n） | 5.55e-17 / 1.11e-16 |
| 手写 GRU vs `nn.GRU` max&#124;diff&#124; | 1.11e-16 |
| 错误 gate 顺序 `(f,i,o,g)` 的 h 偏差 | 4.741e-01 |
| 只对调 `i` / `f` 的 h 偏差 | 1.227e-01 |
| GRU `n` 门偏置写错的偏差 | 0.1552 |
| 形状：`layers=2, bidir=True` 的 out / h / c | (4,10,16) / (4,4,8) / (4,4,8) |
| RNN / LSTM / GRU / LSTM 双向 测试 MAE | 0.259707 / 0.215303 / 0.223200 / 0.225713 |
| T=160 首末梯度比：RNN(ρ=0.9) / LSTM 默认 | 9.615e-13 / 3.952e-34 |
| 遗忘门偏置 1 / 2 / 3 的首末比 | 3.218e-12 / 1.053e-03 / 3.004e-01 |
| 遗忘门偏置 3 的平均每步因子 | 0.9925 |